In [1]:
import matplotlib.pyplot as plt
import numpy as np
import os
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from tensorflow.keras import layers, models
import pickle


In [2]:
data_dir = "../datasets"
x_list = []
y_list = []
batch_size = 32

# Cargar dataset

In [3]:
for file in os.listdir(data_dir):
    path = os.path.join(data_dir, file)
    with open(path, 'rb') as fo:
        data_dict = pickle.load(fo, encoding='bytes')

        x_list.append(data_dict[b'data'])
        y_list.extend(data_dict[b'labels'])

FileNotFoundError: [WinError 3] El sistema no puede encontrar la ruta especificada: '../datasets'

In [ ]:
X = np.concatenate(x_list)
Y = np.array(y_list)

In [ ]:
X = X.reshape(-1, 3,32,32)
X = X.transpose(0,2,3,1)
X = X.astype(float) / 255.0

In [ ]:
y = to_categorical(Y, 10)

In [ ]:
dataset = tf.data.Dataset.from_tensor_slices((X, y))

dataset = dataset.shuffle(buffer_size=len(X), seed=42)

In [ ]:
train_size = int(0.8*len(X))

In [ ]:
train_dataset = dataset.take(train_size)
val_dataset = dataset.skip(train_size)

In [ ]:
train_dataset = train_dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)
val_dataset = val_dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)

In [ ]:
print(len(X), X.shape)
print(len(y), y.shape if isinstance(y, np.ndarray) else 'no es ndarray')



50000 (50000, 32, 32, 3)
50000 (50000, 10)


In [ ]:
IMG_SIZE = (32,32) + (3,)

In [ ]:
base_model = tf.keras.applications.MobileNetV2(input_shape=IMG_SIZE, include_top=False, weights='imagenet')

C:\Users\xavif\AppData\Local\Temp\ipykernel_22992\846687965.py:1: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = tf.keras.applications.MobileNetV2(input_shape=IMG_SIZE, include_top=False, weights='imagenet')


In [ ]:
base_model.summary()

# Congelación de capas

In [ ]:
base_model.trainable = False

In [ ]:
x = base_model.output
x = layers.GlobalAveragePooling2D()(x)
x= layers.Dense(128, activation='relu')(x)
x = layers.Dense(64, activation='relu')(x)
x = layers.Dense(32, activation='relu')(x)
output = layers.Dense(10, activation='softmax')(x)

In [ ]:
model = models.Model(inputs=base_model.input, outputs=output)

In [ ]:
model.compile(
    optimizer='adam',
    loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

In [ ]:
history = model.fit(
    train_dataset, 
    validation_data=val_dataset,
    epochs=10
)

Epoch 1/10


c:\Users\xavif\Documents\MASTER\MobileNet-TransferLearning\.venv\Lib\site-packages\keras\src\backend\tensorflow\nn.py:1172: UserWarning: "`categorical_crossentropy` received `from_logits=True`, but the `output` argument was produced by a Softmax activation and thus does not represent logits. Was this intended?
  output, from_logits = _get_logits(


1250/1250 ━━━━━━━━━━━━━━━━━━━━ 56s 41ms/step - accuracy: 0.2930 - loss: 1.9407 - val_accuracy: 0.3348 - val_loss: 1.8181
Epoch 2/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 39s 31ms/step - accuracy: 0.3381 - loss: 1.8193 - val_accuracy: 0.3371 - val_loss: 1.7986
Epoch 3/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 70s 56ms/step - accuracy: 0.3531 - loss: 1.7784 - val_accuracy: 0.3728 - val_loss: 1.7367
Epoch 4/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 58s 47ms/step - accuracy: 0.3606 - loss: 1.7505 - val_accuracy: 0.3771 - val_loss: 1.7198
Epoch 5/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 58s 47ms/step - accuracy: 0.3720 - loss: 1.7289 - val_accuracy: 0.3824 - val_loss: 1.7117
Epoch 6/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 58s 47ms/step - accuracy: 0.3770 - loss: 1.7135 - val_accuracy: 0.3859 - val_loss: 1.6802
Epoch 7/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 53s 42ms/step - accuracy: 0.3854 - loss: 1.6928 - val_accuracy: 0.3868 - val_loss: 1.6704
Epoch 8/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 38s 31ms/step - accuracy: 0.3931 - loss: 1.67

In [ ]:
plt.plot(history.history['accuracy'], label='accuracy')
plt.plot(history.history['val_accuracy'], label = 'val_accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.ylim(0.5, 1)
plt.legend(loc='lower right')
plt.show()